In [16]:
from pathlib import Path
import sys
import time

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.train import build_logistic_regression_pipeline

In [17]:
TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "train_clean.csv"
VALIDATION_PATH = PROJECT_ROOT / "data" / "processed" / "validation_clean.csv"

train_df = pd.read_csv(TRAIN_PATH)
validation_df = pd.read_csv(VALIDATION_PATH)

print("Training shape:", train_df.shape)
print("Validation shape:", validation_df.shape)

display(train_df.head())

Training shape: (69384, 4)
Validation shape: (1000, 4)


,tweet_id,entity,sentiment,text
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,i am coming to the borders and i will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


In [18]:
X_train = train_df["text"]
y_train = train_df["sentiment"]

X_validation = validation_df["text"]
y_validation = validation_df["sentiment"]

print("Training samples:", len(X_train))
print("Validation samples:", len(X_validation))

Training samples: 69384
Validation samples: 1000


In [19]:
baseline_model = build_logistic_regression_pipeline()

baseline_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",False
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",<function nor...x78249f177f60>
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",<function tok...x78249e45dda0>
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",None
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"max_df max_df: float or int, default=1.0When building the vocabulary ignore terms that have a documentfrequency strictly higher than the given threshold (corpus-specificstop words).If float in range [0.0, 1.0], the parameter represents a proportion ofdocuments, integer absolute counts.This parameter is ignored if vocabulary is not None.",0.98
,"min_df min_df: float or int, default=1When building the vocabulary ignore terms that have a documentfrequency strictly lower than the given threshold. This value is alsocalled cut-off in the literature.If float in range of [0.0, 1.0], the parameter represents a proportionof documents, integer absolute counts.This parameter is ignored 

In [20]:
print(baseline_model.named_steps.keys())

dict_keys(['tfidf', 'classifier'])


In [21]:
start_time = time.perf_counter()

baseline_model.fit(X_train, y_train)

training_time = time.perf_counter() - start_time

print(f"Training time: {training_time:.2f} seconds")

Training time: 40.79 seconds


In [22]:
start_time = time.perf_counter()

validation_predictions = baseline_model.predict(X_validation)

prediction_time = time.perf_counter() - start_time

print(f"Prediction time: {prediction_time:.4f} seconds")

Prediction time: 0.2237 seconds


In [23]:
accuracy = accuracy_score(
    y_validation,
    validation_predictions,
)

macro_precision, macro_recall, macro_f1, _ = (
    precision_recall_fscore_support(
        y_validation,
        validation_predictions,
        average="macro",
        zero_division=0,
    )
)

weighted_precision, weighted_recall, weighted_f1, _ = (
    precision_recall_fscore_support(
        y_validation,
        validation_predictions,
        average="weighted",
        zero_division=0,
    )
)

baseline_metrics = pd.DataFrame(
    {
        "metric": [
            "accuracy",
            "macro_precision",
            "macro_recall",
            "macro_f1",
            "weighted_precision",
            "weighted_recall",
            "weighted_f1",
            "training_time_seconds",
            "prediction_time_seconds",
        ],
        "value": [
            accuracy,
            macro_precision,
            macro_recall,
            macro_f1,
            weighted_precision,
            weighted_recall,
            weighted_f1,
            training_time,
            prediction_time,
        ],
    }
)

baseline_metrics

,metric,value
0,accuracy,0.968000
1,macro_precision,0.969415
2,macro_recall,0.965827
3,macro_f1,0.967366
4,weighted_precision,0.968394
5,weighted_recall,0.968000
6,weighted_f1,0.967961
7,training_time_seconds,40.785787
8,prediction_time_seconds,0.223749


In [24]:
report = classification_report(
    y_validation,
    validation_predictions,
    zero_division=0,
)

print(report)

              precision    recall  f1-score   support

  Irrelevant       0.98      0.94      0.96       172
    Negative       0.95      0.99      0.97       266
     Neutral       0.98      0.95      0.97       285
    Positive       0.96      0.98      0.97       277

    accuracy                           0.97      1000
   macro avg       0.97      0.97      0.97      1000
weighted avg       0.97      0.97      0.97      1000



In [25]:
labels = sorted(y_validation.unique())

confusion = confusion_matrix(
    y_validation,
    validation_predictions,
    labels=labels,
)

confusion_df = pd.DataFrame(
    confusion,
    index=[f"actual_{label}" for label in labels],
    columns=[f"predicted_{label}" for label in labels],
)

confusion_df

,predicted_Irrelevant,predicted_Negative,predicted_Neutral,predicted_Positive
actual_Irrelevant,162,4,1,5
actual_Negative,0,263,2,1
actual_Neutral,1,7,272,5
actual_Positive,2,2,2,271


In [26]:
prediction_results = validation_df[
    ["tweet_id", "entity", "text", "sentiment"]
].copy()

prediction_results["predicted_sentiment"] = (
    validation_predictions
)

prediction_results["correct"] = (
    prediction_results["sentiment"]
    == prediction_results["predicted_sentiment"]
)

mistakes = prediction_results[
    ~prediction_results["correct"]
]

print("Correct predictions:", prediction_results["correct"].sum())
print("Incorrect predictions:", len(mistakes))

display(mistakes.head(20))

Correct predictions: 968
Incorrect predictions: 32


,tweet_id,entity,text,sentiment,predicted_sentiment,correct
68,1908,CallOfDutyBlackopsColdWar,seems like playstation has the marketing deal ...,Neutral,Positive,False
91,3286,Facebook,leaked memo excoriates facebook’s ‘slapdash an...,Negative,Neutral,False
98,3526,Facebook,our hisaperth obiawards ceremony is taking pla...,Neutral,Positive,False
133,9619,PlayStation5(PS5),this playstation5 pre order is an absolute.......,Positive,Neutral,False
215,6675,Fortnite,thank you 👍 fortnite xboxshare pic.twitter.com...,Irrelevant,Positive,False
314,2191,CallOfDuty,whos ready for some zombie royale warzone stre...,Neutral,Irrelevant,False
320,6291,FIFA,worldcupathome: five african matches you would...,Positive,Irrelevant,False
347,10412,RedDeadRedemption(RDR),red dead redemption 2 load times xbox series x...,Irrelevant,Positive,False
390,5149,GrandTheftAuto(GTA),mori😻😻😻😻,Neutral,Negative,False
405,6992,johnson&johnson,"by combining product engagement analytics, dig...",Positive,Neutral,False


In [27]:
RESULTS_DIR = PROJECT_ROOT / "reports" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [28]:
report_dict = classification_report(
    y_validation,
    validation_predictions,
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report_dict).transpose()

report_df.to_csv(
    RESULTS_DIR / "baseline_logistic_regression_report.csv"
)

In [1]:
from pathlib import Path
import sys
import time

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.train import build_svm_pipeline

In [3]:
TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "train_clean.csv"
VALIDATION_PATH = PROJECT_ROOT / "data" / "processed" / "validation_clean.csv"

train_df = pd.read_csv(TRAIN_PATH)
validation_df = pd.read_csv(VALIDATION_PATH)

print("Training shape:", train_df.shape)
print("Validation shape:", validation_df.shape)

Training shape: (69384, 4)
Validation shape: (1000, 4)


In [5]:
X_train = train_df["text"]
y_train = train_df["sentiment"]

X_validation = validation_df["text"]
y_validation = validation_df["sentiment"]

print("Training samples:", len(X_train))
print("Validation samples:", len(X_validation))

Training samples: 69384
Validation samples: 1000


In [6]:
svm_model = build_svm_pipeline()

svm_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",False
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",<function nor...x76bebd97c0e0>
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",<function tok...x76bebca35f80>
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",None
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"max_df max_df: float or int, default=1.0When building the vocabulary ignore terms that have a documentfrequency strictly higher than the given threshold (corpus-specificstop words).If float in range [0.0, 1.0], the parameter represents a proportion ofdocuments, integer absolute counts.This parameter is ignored if vocabulary is not None.",0.98
,"min_df min_df: float or int, default=1When building the vocabulary ignore terms that have a documentfrequency strictly lower than the given threshold. This value is alsocalled cut-off in the literature.If float in range of [0.0, 1.0], the parameter represents a proportionof documents, integer absolute counts.This parameter is ignored 

In [8]:
start_time = time.perf_counter()

svm_model.fit(X_train, y_train)

training_time = time.perf_counter() - start_time

print(f"Training time: {training_time:.2f} seconds")

Training time: 9.38 seconds


In [9]:
start_time = time.perf_counter()

validation_predictions = svm_model.predict(X_validation)

prediction_time = time.perf_counter() - start_time

print(f"Prediction time: {prediction_time:.4f} seconds")

Prediction time: 0.1284 seconds


In [10]:
accuracy = accuracy_score(
    y_validation,
    validation_predictions,
)

macro_precision, macro_recall, macro_f1, _ = (
    precision_recall_fscore_support(
        y_validation,
        validation_predictions,
        average="macro",
        zero_division=0,
    )
)

weighted_precision, weighted_recall, weighted_f1, _ = (
    precision_recall_fscore_support(
        y_validation,
        validation_predictions,
        average="weighted",
        zero_division=0,
    )
)

svm_metrics = pd.DataFrame(
    {
        "metric": [
            "accuracy",
            "macro_precision",
            "macro_recall",
            "macro_f1",
            "weighted_precision",
            "weighted_recall",
            "weighted_f1",
            "training_time_seconds",
            "prediction_time_seconds",
        ],
        "value": [
            accuracy,
            macro_precision,
            macro_recall,
            macro_f1,
            weighted_precision,
            weighted_recall,
            weighted_f1,
            training_time,
            prediction_time,
        ],
    }
)

svm_metrics

,metric,value
0,accuracy,0.977000
1,macro_precision,0.977205
2,macro_recall,0.977255
3,macro_f1,0.977047
4,weighted_precision,0.977401
5,weighted_recall,0.977000
6,weighted_f1,0.976995
7,training_time_seconds,9.383048
8,prediction_time_seconds,0.128422


In [11]:
report = classification_report(
    y_validation,
    validation_predictions,
    zero_division=0,
)

print(report)

              precision    recall  f1-score   support

  Irrelevant       0.98      0.98      0.98       172
    Negative       0.98      0.99      0.98       266
     Neutral       1.00      0.95      0.97       285
    Positive       0.96      0.99      0.97       277

    accuracy                           0.98      1000
   macro avg       0.98      0.98      0.98      1000
weighted avg       0.98      0.98      0.98      1000



In [13]:
labels = sorted(y_validation.unique())

confusion = confusion_matrix(
    y_validation,
    validation_predictions,
    labels=labels,
)

confusion_df = pd.DataFrame(
    confusion,
    index=[f"actual_{label}" for label in labels],
    columns=[f"predicted_{label}" for label in labels],
)

confusion_df

,predicted_Irrelevant,predicted_Negative,predicted_Neutral,predicted_Positive
actual_Irrelevant,168,1,0,3
actual_Negative,1,263,1,1
actual_Neutral,2,3,272,8
actual_Positive,1,2,0,274


In [14]:
prediction_results = validation_df[
    ["tweet_id", "entity", "text", "sentiment"]
].copy()

prediction_results["predicted_sentiment"] = (
    validation_predictions
)

prediction_results["correct"] = (
    prediction_results["sentiment"]
    == prediction_results["predicted_sentiment"]
)

mistakes = prediction_results[
    ~prediction_results["correct"]
]

print("Correct predictions:", prediction_results["correct"].sum())
print("Incorrect predictions:", len(mistakes))

display(mistakes.head(20))

Correct predictions: 977
Incorrect predictions: 23


,tweet_id,entity,text,sentiment,predicted_sentiment,correct
91,3286,Facebook,leaked memo excoriates facebook’s ‘slapdash an...,Negative,Neutral,False
98,3526,Facebook,our hisaperth obiawards ceremony is taking pla...,Neutral,Positive,False
109,10585,RedDeadRedemption(RDR),red dead redemption 2 rdonline rdr2 dailychall...,Neutral,Positive,False
215,6675,Fortnite,thank you 👍 fortnite xboxshare pic.twitter.com...,Irrelevant,Positive,False
255,12480,WorldOfCraft,zysola.blogspot.com/p/welcome.html… zysola tec...,Neutral,Irrelevant,False
347,10412,RedDeadRedemption(RDR),red dead redemption 2 load times xbox series x...,Irrelevant,Positive,False
390,5149,GrandTheftAuto(GTA),mori😻😻😻😻,Neutral,Negative,False
392,10095,PlayerUnknownsBattlegrounds(PUBG),pubg is no more available on android playstore...,Negative,Positive,False
426,9116,Nvidia,nice 👍 follow 👉 👈 twitter 👉 👈 youtube 👉youtube...,Neutral,Irrelevant,False
443,800,ApexLegends,modernwarfare sd the best fps out! was in love...,Positive,Irrelevant,False


In [15]:
RESULTS_DIR = PROJECT_ROOT / "reports" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

svm_metrics.to_csv(
    RESULTS_DIR / "baseline_svm_metrics.csv",
    index=False,
)

In [16]:
prediction_results.to_csv(
    RESULTS_DIR / "baseline_svm_predictions.csv",
    index=False,
)

report_dict = classification_report(
    y_validation,
    validation_predictions,
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report_dict).transpose()

report_df.to_csv(
    RESULTS_DIR / "baseline_svm_report.csv"
)

In [3]:
from pathlib import Path
import sys
import time
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.train import build_svm_pipeline

In [4]:
comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Weighted F1",
        "Training Time (s)",
        "Prediction Time (s)",
        "Correct Predictions",
        "Incorrect Predictions",
    ],
    "Logistic Regression": [
        0.9680,
        0.9694,
        0.9658,
        0.9674,
        0.9680,
        43.10,
        0.116,
        968,
        32,
    ],
    "Linear SVM": [
        0.9770,
        0.9772,
        0.9773,
        0.9770,
        0.9770,
        9.38,
        0.128,
        977,
        23,
    ],
})

comparison

,Metric,Logistic Regression,Linear SVM
0,Accuracy,0.9680,0.9770
1,Macro Precision,0.9694,0.9772
2,Macro Recall,0.9658,0.9773
3,Macro F1,0.9674,0.9770
4,Weighted F1,0.9680,0.9770
5,Training Time (s),43.1000,9.3800
6,Prediction Time (s),0.1160,0.1280
7,Correct Predictions,968.0000,977.0000
8,Incorrect Predictions,32.0000,23.0000


In [6]:
comparison.to_csv(
    PROJECT_ROOT / "reports" / "results" / "baseline_model_comparison.csv",
    index=False,
)